# Step 3: Download and Understand NASA Battery Dataset

This notebook downloads the NASA Battery Dataset and performs initial data exploration.

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Download latest version
path = kagglehub.dataset_download("mystifoe77/nasa-battery-data-cleaned")

print("Path to dataset files:", path)

# List files in the dataset
files = os.listdir(path)
print("Files in dataset:", files)

In [ ]:
# Load the dataset
df = pd.read_csv('../data/raw/Battery_Data_Cleaned.csv')
print("Dataset shape:", df.shape)
print("Columns:", list(df.columns))
print(df.head())

In [ ]:
# Calculate rated capacity (max capacity per battery)
rated_capacities = df.groupby('battery_id')['Capacity'].max()
df['rated_capacity'] = df['battery_id'].map(rated_capacities)

# Calculate SoH
df['soh'] = (df['Capacity'] / df['rated_capacity']) * 100

# Create the required dataframe
soh_df = df[['battery_id', 'uid', 'Capacity', 'soh']].rename(columns={'uid': 'cycle', 'Capacity': 'capacity'})

print("SoH DataFrame:")
print(soh_df.head())
print("Number of batteries:", soh_df['battery_id'].nunique())
print("Cycles per battery:")
print(soh_df.groupby('battery_id')['cycle'].count())

In [ ]:
# EDA
print("Missing values:")
print(df.isnull().sum())

print("SoH statistics:")
print(soh_df['soh'].describe())

# Plot 1: Cycle vs Capacity for a sample battery
sample_battery = soh_df['battery_id'].iloc[0]
sample_data = soh_df[soh_df['battery_id'] == sample_battery]

plt.figure(figsize=(10, 6))
plt.plot(sample_data['cycle'], sample_data['capacity'], marker='o')
plt.xlabel('Cycle')
plt.ylabel('Capacity')
plt.title(f'Cycle vs Capacity for Battery {sample_battery}')
plt.grid(True)
plt.savefig('../outputs/plots/cycle_vs_capacity.png')
plt.show()

# Plot 2: Cycle vs SoH
plt.figure(figsize=(10, 6))
plt.plot(sample_data['cycle'], sample_data['soh'], marker='o', color='orange')
plt.xlabel('Cycle')
plt.ylabel('SoH (%)')
plt.title(f'Cycle vs SoH for Battery {sample_battery}')
plt.grid(True)
plt.savefig('../outputs/plots/cycle_vs_soh.png')
plt.show()

# Plot 3: Histogram of SoH
plt.figure(figsize=(8, 6))
plt.hist(soh_df['soh'], bins=20, edgecolor='black')
plt.xlabel('SoH (%)')
plt.ylabel('Frequency')
plt.title('Histogram of SoH')
plt.savefig('../outputs/plots/soh_histogram.png')
plt.show()